<a href="https://colab.research.google.com/github/c81e32cat/kunteynir/blob/deepmultilingualpunctuation/kunteynir_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas

Тексты, выгруженные с genius (при помощи `lyricsgenius` - библиотека не работает из google colab), приводим в нормальный вид (убираем аннотации, ссылки на обложки и т. д.)

In [ ]:
import pandas as pd
import json
import re

with open('/content/Lyrics_Kunteynir_Fixed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

clean_data = []
for song in data['songs']:
    clean_data.append({
        'title': song['title'],
        'lyrics': song['lyrics']
    })

for i in range(len(clean_data)):
    curr_text=clean_data[i]['lyrics'].split('\n')
    new_text = []
    for string in curr_text:
        if string and string[0]!='—' and string not in new_text:
            new_text.append(string)
    new_text = '\n'.join(new_text)
    new_text = re.sub(r'\(.*?\)', '', new_text)
    clean_data[i]['lyrics']=new_text

df = pd.DataFrame(clean_data)

print(df.head())

In [ ]:
!pip install deepmultilingualpunctuation
!pip install transformers==4.30.2

In [ ]:
import sys
from transformers import pipeline
import deepmultilingualpunctuation

#возникли проблемы с transformers, так что gemini помог мне запустить библиотеку
def fixed_init(self, model='oliverguhr/fullstop-punctuation-multilang-large', device=None):
    import torch
    if device is None:
        device = 0 if torch.cuda.is_available() else -1

    self.pipe = pipeline('ner', model=model, device=device)

deepmultilingualpunctuation.PunctuationModel.__init__ = fixed_init

from deepmultilingualpunctuation import PunctuationModel
model = PunctuationModel()

In [ ]:
dmp_data=[]

for i in range(len(clean_data)):
  text = clean_data[i]['lyrics'].replace('\n', ', ').lower()
  result = model.restore_punctuation(text)
  dmp_data.append({
        'title': clean_data[i]['title'],
        'lyrics': result
    })
  #print(dmp_data[i]['lyrics'].replace('.', '.\n'))